# 🚦 Road Sign Detection – Dataset & Model Exploration
Notebook do eksploracji datasetu GTSDB, weryfikacji labelek i wizualizacji predykcji modelu YOLOv11.
**Wymagania:** uruchom najpierw `make ml-prepare` aby pobrać i skonwertować dataset.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATASET_DIR = PROJECT_ROOT / "datasets" / "gtsdb"
WEIGHTS_PATH = PROJECT_ROOT / "runs" / "detect" / "train" / "weights" / "best.pt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset dir:  {DATASET_DIR}")
print(f"Dataset exists: {DATASET_DIR.exists()}")

Project root: /home/matiar/Programming/engin33ring-thesis/ml
Dataset dir:  /home/matiar/Programming/engin33ring-thesis/ml/datasets/gtsdb
Dataset exists: False


## 1. 📊 Przegląd mapowania klas

In [3]:
from ml.src.label_mapping import (
    GTSDB_FINE_NAMES,
    GTSDB_CLASS_TO_SUPERCATEGORY,
    SUPERCATEGORY_NAMES,
    fine_class_name,
)
import pandas as pd

rows = []
for cls_id, name in sorted(GTSDB_FINE_NAMES.items()):
    super_id = GTSDB_CLASS_TO_SUPERCATEGORY[cls_id]
    rows.append({
        "class_id": cls_id,
        "name": name,
        "super_id": super_id,
        "super_name": SUPERCATEGORY_NAMES[super_id],
    })

df_classes = pd.DataFrame(rows)
print(f"Liczba klas: {len(df_classes)}")
print(df_classes.groupby("super_name")["name"].count())
df_classes

Liczba klas: 43
super_name
danger         15
mandatory      10
other           2
prohibitory    16
Name: name, dtype: int64


,class_id,name,super_id,super_name
0,0,speed_limit_20,0,prohibitory
1,1,speed_limit_30,0,prohibitory
2,2,speed_limit_50,0,prohibitory
3,3,speed_limit_60,0,prohibitory
4,4,speed_limit_70,0,prohibitory
5,5,speed_limit_80,0,prohibitory
6,6,end_speed_limit_80,0,prohibitory
7,7,speed_limit_100,0,prohibitory
8,8,speed_limit_120,0,prohibitory
9,9,no_overtaking,0,prohibitory


## 2. 🖼️ Przegląd datasetu

In [4]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

train_images = sorted((DATASET_DIR / "images" / "train").glob("*"))
val_images = sorted((DATASET_DIR / "images" / "val").glob("*"))

print(f"Obrazy treningowe: {len(train_images)}")
print(f"Obrazy walidacyjne: {len(val_images)}")
print(f"Łącznie: {len(train_images) + len(val_images)}")

Obrazy treningowe: 0
Obrazy walidacyjne: 0
Łącznie: 0


### 2.1 Przykładowe obrazy (bez adnotacji)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Przykładowe obrazy z datasetu GTSDB", fontsize=16)

sample_images = train_images[:8]

for ax, img_path in zip(axes.flat, sample_images):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

### 2.2 Obrazy z nałożonymi labelkami YOLO

In [ ]:
from ml.src.visualize import draw_yolo_boxes

# Znajdź obrazy, które mają niepuste labelki
annotated_samples = []
for img_path in train_images:
    label_path = DATASET_DIR / "labels" / "train" / f"{img_path.stem}.txt"
    if label_path.exists() and label_path.read_text().strip():
        annotated_samples.append((img_path, label_path))
    if len(annotated_samples) >= 8:
        break

print(f"Znaleziono {len(annotated_samples)} obrazów z adnotacjami (pokazuję 8)")

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Obrazy z bounding boxami (YOLO labels)", fontsize=16)

for ax, (img_path, label_path) in zip(axes.flat, annotated_samples):
    img = cv2.imread(str(img_path))
    img_with_boxes = draw_yolo_boxes(img, label_path, line_thickness=3)
    img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. 📈 Rozkład klas w datasecie

In [ ]:
from collections import Counter

class_counter = Counter()

for split in ["train", "val"]:
    labels_dir = DATASET_DIR / "labels" / split
    if not labels_dir.exists():
        continue
    for label_file in labels_dir.glob("*.txt"):
        for line in label_file.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                class_counter[int(parts[0])] += 1

# Posortuj po class ID
sorted_classes = sorted(class_counter.items())
class_ids = [c[0] for c in sorted_classes]
counts = [c[1] for c in sorted_classes]
names = [GTSDB_FINE_NAMES.get(c, str(c)) for c in class_ids]

# Super-kategoria → kolor
bar_colors = []
color_map = {0: "#e74c3c", 1: "#3498db", 2: "#f39c12", 3: "#95a5a6"}
for cid in class_ids:
    sid = GTSDB_CLASS_TO_SUPERCATEGORY.get(cid, 3)
    bar_colors.append(color_map[sid])

fig, ax = plt.subplots(figsize=(18, 8))
bars = ax.barh(range(len(names)), counts, color=bar_colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("Liczba instancji")
ax.set_title("Rozkład klas w datasecie GTSDB (train + val)")
ax.invert_yaxis()

# Legenda super-kategorii
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e74c3c", label="prohibitory"),
    Patch(facecolor="#3498db", label="mandatory"),
    Patch(facecolor="#f39c12", label="danger"),
    Patch(facecolor="#95a5a6", label="other"),
]
ax.legend(handles=legend_elements, loc="lower right")

# Wartości na końcu słupków
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=7)

plt.tight_layout()
plt.show()

print(f"Liczba klas z danymi: {len(sorted_classes)} / 43")

In [ ]:
widths = []
heights = []
areas = []

for split in ["train", "val"]:
    labels_dir = DATASET_DIR / "labels" / split
    if not labels_dir.exists():
        continue
    for label_file in labels_dir.glob("*.txt"):
        for line in label_file.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                w, h = float(parts[3]), float(parts[4])
                widths.append(w)
                heights.append(h)
                areas.append(w * h)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Rozkład rozmiarów bounding boxów (znormalizowane)", fontsize=14)
axes[0].hist(widths, bins=50, color="#3498db", edgecolor="white")
axes[0].set_title("Szerokość bbox")
axes[0].set_xlabel("Znormalizowana szerokość")

axes[1].hist(heights, bins=50, color="#e74c3c", edgecolor="white")
axes[1].set_title("Wysokość bbox")
axes[1].set_xlabel("Znormalizowana wysokość")

axes[2].scatter(widths, heights, alpha=0.3, s=10, c="#2ecc71")
axes[2].set_title("Szerokość vs Wysokość")
axes[2].set_xlabel("Szerokość")
axes[2].set_ylabel("Wysokość")

plt.tight_layout()
plt.show()

print(f"Mediana szerokości: {np.median(widths):.4f}")
print(f"Mediana wysokości:  {np.median(heights):.4f}")
print(f"Mediana pola:       {np.median(areas):.6f}")

## 5. 🔮 Predykcje modelu

Ta sekcja wymaga wytrenowanego modelu. Uruchom najpierw `make ml-train`."

## 4. 📐 Rozkład rozmiarów bounding boxów

In [ ]:
from ultralytics import YOLO

if not WEIGHTS_PATH.exists():
    print(f"⚠️  Brak wytrenowanego modelu: {WEIGHTS_PATH}")
    print("Uruchom: make ml-train")
    model = None
else:
    model = YOLO(str(WEIGHTS_PATH))
    print(f"✅ Model załadowany: {WEIGHTS_PATH}")
    print(f"   Klasy: {model.names}")

### 5.1 Predykcje na zbiorze walidacyjnym

In [ ]:
from ml.src.visualize import visualise_predictions

val_annotated = None
if WEIGHTS_PATH.exists():
    # Wybierz obrazy z labelkami (żeby było co porównać)
    val_annotated = []
    for img_path in val_images:
        label_path = DATASET_DIR / "labels" / "val" / f"{img_path.stem}.txt"
        if label_path.exists() and label_path.read_text().strip():
            val_annotated.append(img_path)
        if len(val_annotated) >= 6:
            break

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle("Predykcje modelu na zbiorze walidacyjnym", fontsize=16)

    for ax, img_path in zip(axes.flat, val_annotated):
        img = cv2.imread(str(img_path))
        results = model.predict(img, imgsz=640, conf=0.25, verbose=False)
        img_pred = visualise_predictions(img, results, confidence_threshold=0.25)
        img_rgb = cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        n_detections = sum(len(r.boxes) for r in results if r.boxes is not None)
        ax.set_title(f"{img_path.name} ({n_detections} det.)", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Pomiń — brak wytrenowanego modelu.")

In [ ]:
if WEIGHTS_PATH.exists() and val_annotated:
    fig, axes = plt.subplots(3, 2, figsize=(20, 18))
    fig.suptitle("Ground Truth (lewo) vs Predykcja modelu (prawo)", fontsize=16)

    for row, img_path in zip(axes, val_annotated[:3]):
        img = cv2.imread(str(img_path))
        label_path = DATASET_DIR / "labels" / "val" / f"{img_path.stem}.txt"

        # Ground truth
        img_gt = draw_yolo_boxes(img, label_path, line_thickness=3)
        img_gt_rgb = cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB)
        row[0].imshow(img_gt_rgb)
        row[0].set_title(f"GT: {img_path.name}", fontsize=10)
        row[0].axis("off")

        # Prediction
        results = model.predict(img, imgsz=640, conf=0.25, verbose=False)
        img_pred = visualise_predictions(img, results, confidence_threshold=0.25)
        img_pred_rgb = cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB)
        row[1].imshow(img_pred_rgb)
        n_det = sum(len(r.boxes) for r in results if r.boxes is not None)
        row[1].set_title(f"Pred: {img_path.name} ({n_det} det.)", fontsize=10)
        row[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Pomiń — brak wytrenowanego modelu.")

### 5.3 Confidence histogram per klasa (top-10)

In [ ]:
if WEIGHTS_PATH.exists():
    from collections import defaultdict

    conf_per_class = defaultdict(list)

    # Zbierz predykcje z całego zbioru walidacyjnego
    for img_path in val_images[:100]:  # max. 100 obrazów
        img = cv2.imread(str(img_path))
        results = model.predict(img, imgsz=640, conf=0.1, verbose=False)
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                cls_name = GTSDB_FINE_NAMES.get(cls_id, str(cls_id))
                conf_per_class[cls_name].append(conf)

    # Top 10 klas z największą liczbą detekcji
    top_classes = sorted(conf_per_class.items(), key=lambda x: len(x[1]), reverse=True)[:10]

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle("Rozkład confidence per klasa (top 10)", fontsize=14)

    for ax, (cls_name, confs) in zip(axes.flat, top_classes):
        ax.hist(confs, bins=20, color="#2ecc71", edgecolor="white", range=(0, 1))
        ax.set_title(f"{cls_name}")
        ax.set_xlim(0, 1)
        ax.axvline(x=0.25, color="red", linestyle="--", linewidth=0.8)

    plt.tight_layout()
    plt.show()

    print(f"Łączna liczba detekcji: {sum(len(v) for v in conf_per_class.values())}")
    print(f"Klasy z detekcjami: {len(conf_per_class)}")
else:
    print("⚠️  Pomiń — brak wytrenowanego modelu.")

## 6. 🧪 Inferencja ONNX

Po eksporcie (`make ml-export`) sprawdzamy, że model ONNX daje te same wyniki.

In [ ]:
onnx_path = WEIGHTS_PATH.with_suffix(".onnx")

if onnx_path.exists():
    import onnx

    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print(f"✅ ONNX model valid: {onnx_path}")

    # Sprawdzenie wejścia/wyjścia
    for inp in onnx_model.graph.input:
        shape = [d.dim_value or d.dim_param for d in inp.type.tensor_type.shape.dim]
        print(f"   Input:  {inp.name} → {shape}")
    for out in onnx_model.graph.output:
        shape = [d.dim_value or d.dim_param for d in out.type.tensor_type.shape.dim]
        print(f"   Output: {out.name} → {shape}")
else:
    print(f"⚠️  Brak modelu ONNX: {onnx_path}")
    print("Uruchom: make ml-export")

In [ ]:
if onnx_path.exists():
    import onnxruntime as ort

    session = ort.InferenceSession(str(onnx_path))

    # Predykcja przez ONNX Runtime — Ultralytics obsługuje to transparentnie
    onnx_model_ul = YOLO(str(onnx_path))

    sample_img_path = val_annotated[0] if val_annotated else val_images[0]
    sample_img = cv2.imread(str(sample_img_path))

    results_onnx = onnx_model_ul.predict(sample_img, imgsz=640, conf=0.25, verbose=False)

    img_onnx = visualise_predictions(sample_img, results_onnx)
    img_onnx_rgb = cv2.cvtColor(img_onnx, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    ax.imshow(img_onnx_rgb)
    ax.set_title("Predykcja ONNX Runtime", fontsize=14)
    ax.axis("off")
    plt.show()

    n_det = sum(len(r.boxes) for r in results_onnx if r.boxes is not None)
    print(f"Detekcje ONNX: {n_det}")
else:
    print("⚠️  Pomiń — brak modelu ONNX.")

## 7. 📋 Podsumowanie

| Krok | Komenda |
|---|---|
| Pobranie datasetu | `make ml-prepare` |
| Trening | `make ml-train` |
| Ewaluacja | `make ml-evaluate` |
| Eksport ONNX | `make ml-export` |"